# Gold Layer — Revenue, Occupancy & Guest Experience KPIs
Produces 6 gold tables: daily revenue, occupancy analysis, loyalty segments, channel performance, weekly trends, and property scorecards.

In [ ]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, round as spark_round,
    countDistinct, when, lit, date_trunc, weekofyear, year, month,
    dayofweek, min as spark_min, max as spark_max, current_timestamp
)
from pyspark.sql import Window
import pyspark.sql.functions as F

df_bk   = spark.read.format('delta').table('silver_bookings')
df_prop = spark.read.format('delta').table('silver_properties')
df_gst  = spark.read.format('delta').table('silver_guests')
df_rev  = spark.read.format('delta').table('silver_reviews')

# Derive a per-booking repeat-guest flag (a guest who appears in more than one
# booking). The Gold loyalty/scorecard tables aggregate avg('is_repeat_guest'),
# so this column must exist on the bookings dataframe.
df_bk = df_bk.withColumn(
    'is_repeat_guest',
    (count('booking_id').over(Window.partitionBy('guest_id')) > 1).cast('int'),
)


In [ ]:
# Gold 1 — Daily revenue summary per property
daily_rev = (
    df_bk.filter(col('is_completed') == 1)
    .join(df_prop.select('property_id', 'city', 'country', 'star_rating', 'room_count'), 'property_id', 'left')
    .groupBy('property_id', 'city', 'country', 'star_rating', 'room_count', 'check_in_date')
    .agg(
        count('booking_id').alias('bookings'),
        spark_sum('revenue').alias('total_revenue'),
        spark_round(avg('room_rate'), 2).alias('adr'),
        spark_round(avg('length_of_stay'), 2).alias('avg_los'),
    )
    .withColumn('revpar',
        spark_round(col('total_revenue') / col('room_count'), 2)
    )
    .withColumn('occupancy_rate',
        spark_round(col('bookings') / col('room_count') * 100, 2)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
daily_rev.write.mode('overwrite').format('delta').saveAsTable('gold_daily_revenue')
print(f'Gold daily revenue: {spark.table("gold_daily_revenue").count()} rows')

In [ ]:
# Gold 2 — Occupancy analysis (room type × property)
occupancy = (
    df_bk
    .join(df_prop.select('property_id', 'city', 'star_rating', 'room_count'), 'property_id', 'left')
    .groupBy('property_id', 'city', 'star_rating', 'room_type', 'adr_band')
    .agg(
        count('booking_id').alias('total_bookings'),
        spark_sum('is_completed').alias('completed_bookings'),
        spark_sum('is_cancelled').alias('cancelled_bookings'),
        spark_round(spark_sum('revenue'), 2).alias('total_revenue'),
        spark_round(avg('room_rate'), 2).alias('avg_rate'),
        spark_round(avg('length_of_stay'), 2).alias('avg_los'),
    )
    .withColumn('cancellation_rate',
        spark_round(col('cancelled_bookings') / col('total_bookings') * 100, 2)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
occupancy.write.mode('overwrite').format('delta').saveAsTable('gold_occupancy_analysis')
print(f'Gold occupancy analysis: {spark.table("gold_occupancy_analysis").count()} rows')

In [ ]:
# Gold 3 — Loyalty segment performance
loyalty = (
    df_bk.filter(col('is_completed') == 1)
    .join(df_gst.select('guest_id', 'loyalty_tier', 'region', 'age_group'), 'guest_id', 'left')
    .groupBy('loyalty_tier', 'region', 'age_group')
    .agg(
        count('booking_id').alias('total_bookings'),
        countDistinct('guest_id').alias('unique_guests'),
        spark_round(spark_sum('revenue'), 2).alias('total_revenue'),
        spark_round(avg('room_rate'), 2).alias('avg_rate'),
        spark_round(avg('length_of_stay'), 2).alias('avg_los'),
        spark_round(avg('is_repeat_guest') * 100, 1).alias('repeat_guest_pct'),
    )
    .withColumn('revenue_per_guest',
        spark_round(col('total_revenue') / col('unique_guests'), 2)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
loyalty.write.mode('overwrite').format('delta').saveAsTable('gold_loyalty_segments')
print(f'Gold loyalty segments: {spark.table("gold_loyalty_segments").count()} rows')

In [ ]:
# Gold 4 — Channel performance
channel = (
    df_bk
    .groupBy('channel')
    .agg(
        count('booking_id').alias('total_bookings'),
        spark_sum('is_completed').alias('completed'),
        spark_sum('is_cancelled').alias('cancelled'),
        spark_round(spark_sum('revenue'), 2).alias('total_revenue'),
        spark_round(avg('room_rate'), 2).alias('avg_rate'),
        spark_round(avg('length_of_stay'), 2).alias('avg_los'),
    )
    .withColumn('conversion_rate',
        spark_round(col('completed') / col('total_bookings') * 100, 2)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
channel.write.mode('overwrite').format('delta').saveAsTable('gold_channel_performance')
print(f'Gold channel performance: {spark.table("gold_channel_performance").count()} rows')

In [ ]:
# Gold 5 — Weekly trends
weekly = (
    df_bk.filter(col('is_completed') == 1)
    .withColumn('week', date_trunc('week', col('check_in_date')))
    .groupBy('week')
    .agg(
        count('booking_id').alias('bookings'),
        spark_round(spark_sum('revenue'), 2).alias('total_revenue'),
        spark_round(avg('room_rate'), 2).alias('adr'),
        spark_round(avg('length_of_stay'), 2).alias('avg_los'),
    )
    .orderBy('week')
    .withColumn('gold_timestamp', current_timestamp())
)
weekly.write.mode('overwrite').format('delta').saveAsTable('gold_weekly_trends')
print(f'Gold weekly trends: {spark.table("gold_weekly_trends").count()} rows')

In [ ]:
# Gold 6 — Property scorecards
review_agg = (
    df_rev
    .groupBy('property_id')
    .agg(
        count('review_id').alias('review_count'),
        spark_round(avg('overall_score'), 2).alias('avg_overall_score'),
        spark_round(avg('cleanliness_score'), 2).alias('avg_cleanliness'),
        spark_round(avg('service_score'), 2).alias('avg_service'),
        spark_round(avg('value_score'), 2).alias('avg_value'),
        spark_round(avg('food_score'), 2).alias('avg_food'),
    )
)

booking_agg = (
    df_bk.filter(col('is_completed') == 1)
    .groupBy('property_id')
    .agg(
        count('booking_id').alias('total_bookings'),
        spark_round(spark_sum('revenue'), 2).alias('total_revenue'),
        spark_round(avg('room_rate'), 2).alias('adr'),
        spark_round(avg('length_of_stay'), 2).alias('avg_los'),
        spark_round(avg('is_repeat_guest') * 100, 1).alias('repeat_guest_pct'),
    )
)

scorecard = (
    df_prop.select('property_id', 'property_name', 'city', 'country', 'star_rating', 'room_count', 'property_type')
    .join(booking_agg, 'property_id', 'left')
    .join(review_agg,  'property_id', 'left')
    .withColumn('revpar',
        spark_round(col('total_revenue') / col('room_count'), 2)
    )
    .withColumn('performance_band',
        when(col('avg_overall_score') >= 8.5, 'Excellent')
        .when(col('avg_overall_score') >= 7.0, 'Good')
        .when(col('avg_overall_score') >= 5.5, 'Needs Improvement')
        .otherwise('Critical')
    )
    .withColumn('gold_timestamp', current_timestamp())
)
scorecard.write.mode('overwrite').format('delta').saveAsTable('gold_property_scorecards')
print(f'Gold property scorecards: {spark.table("gold_property_scorecards").count()} rows')

In [ ]:
print('Gold aggregation complete.')
for tbl in ['gold_daily_revenue', 'gold_occupancy_analysis', 'gold_loyalty_segments',
            'gold_channel_performance', 'gold_weekly_trends', 'gold_property_scorecards']:
    n = spark.read.format('delta').table(tbl).count()
    print(f'  {tbl:<35} {n:>6,} rows')